# Model Training and Evaluation (Legacy Branch – RBP Only)

## Overview
This notebook corresponds to the **legacy/initial baseline** of the project. It uses only **Relative Band Power (RBP) features** (95 features total) extracted from 19 EEG channels.  
No functional connectivity, feature selection, or nested cross‑validation is applied.  
The purpose of this branch is to establish a minimal, leakage‑free benchmark against which later improvements (PLV, SelectKBest, Nested CV) can be measured.

## Pipeline Summary

1. **Load the Processed Dataset**  
   The feature matrix (`eeg_rbp_features.csv`) generated by the preprocessing notebook is loaded. Metadata (`subject_id`, `epoch_id`, `label`) is separated from the actual features.

2. **Data Encoding & Scaling**  
   - Categorical labels (`'AD'`, `'FTD'`, `'CN'`) are encoded to integers using `LabelEncoder`.  
   - **StandardScaler** is applied **inside each cross‑validation fold** via a `Pipeline`. This prevents information leakage from the test fold into the training fold.

3. **Subject‑Wise Data Splitting (Critical)**  
   - To avoid data leakage, `GroupKFold` (5 folds) is used with `subject_id` as the grouping variable.  
   - A sanity check asserts that **no subject appears in both training and test sets** across all folds.

4. **Model Training**  
   Five classical classifiers are trained and evaluated inside the cross‑validation loop:
   - K‑Nearest Neighbors (KNN)  
   - Decision Tree  
   - Support Vector Machine (SVM)  
   - Logistic Regression  
   - Linear Discriminant Analysis (LDA)

5. **Evaluation Metrics**  
   For each model, we compute:
   - Accuracy  
   - Macro‑averaged Precision, Recall, and F1‑Score (treating all three classes equally)


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn", "joblib"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

PROCESSED_DIR = '../data/processed/'
MODELS_DIR = '../data/processed/models/'
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5  # GroupKFold folds

print('All imports successful.')

In [ ]:
# Load the feature matrix generated in Notebook 01
csv_path = os.path.join(PROCESSED_DIR, 'eeg_rbp_features.csv')
df = pd.read_csv(csv_path)

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['label'].value_counts())
print(f'\nUnique subjects: {df["subject_id"].nunique()}')
df.head()

In [ ]:
# Separate metadata from features and labels
METADATA_COLS = ['subject_id', 'epoch_id', 'label']
FEATURE_COLS  = [c for c in df.columns if c not in METADATA_COLS]

X      = df[FEATURE_COLS].values          # Feature matrix  (n_epochs, n_features)
groups = df['subject_id'].values          # Group vector    (n_epochs,)
y_raw  = df['label'].values               # String labels   (n_epochs,)

print(f'Feature matrix shape : {X.shape}')
print(f'Number of features   : {len(FEATURE_COLS)}')
print(f'Feature columns (first 10): {FEATURE_COLS[:10]}')

In [ ]:
# Encode categorical labels ('AD', 'FTD', 'CN') → integers
le = LabelEncoder()
y  = le.fit_transform(y_raw)

print('Label encoding mapping:')
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {cls} → {idx}')

# Note: StandardScaler is applied inside each cross-validation fold
# (via sklearn Pipeline) to avoid data leakage from the test fold.
print('\nStandardScaler will be applied inside the CV pipeline to prevent leakage.')

In [ ]:
# Configure subject-wise GroupKFold cross-validation
gkf = GroupKFold(n_splits=N_SPLITS)

print(f'GroupKFold with {N_SPLITS} folds')
print(f'Total subjects: {len(np.unique(groups))}')
print(f'~{len(np.unique(groups)) // N_SPLITS} subjects held out per fold')

# Quick sanity check — verify no subject appears in both train and test
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    train_subs = set(groups[train_idx])
    test_subs  = set(groups[test_idx])
    assert len(train_subs & test_subs) == 0, f'Data leakage in fold {fold}!'
    print(f'  Fold {fold+1}: train={len(train_idx)} epochs ({len(train_subs)} subjects) | '
          f'test={len(test_idx)} epochs ({len(test_subs)} subjects)')

print('\n✓ No data leakage detected across all folds.')

In [ ]:
def make_pipeline(classifier):
    """Wraps a classifier in a StandardScaler → Classifier pipeline."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    classifier)
    ])

MODELS = {
    'KNN':                 make_pipeline(KNeighborsClassifier(n_neighbors=5)),
    'Decision Tree':       make_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    'SVM':                 make_pipeline(SVC(kernel='rbf', C=1.0, probability=True,
                                            random_state=RANDOM_STATE)),
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=1000,
                                                             random_state=RANDOM_STATE)),
    'LDA':                 make_pipeline(LinearDiscriminantAnalysis()),
}

SCORING = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

print(f'Models to train: {list(MODELS.keys())}')
print(f'Scoring metrics: {SCORING}')

In [ ]:
# Train all models with GroupKFold cross-validation
cv_results = {}

for model_name, pipeline in MODELS.items():
    print(f'Training {model_name}...', end=' ')
    
    scores = cross_validate(
        pipeline, X, y,
        groups=groups,
        cv=gkf,
        scoring=SCORING,
        return_train_score=False,
        n_jobs=-1
    )
    cv_results[model_name] = scores
    
    mean_acc = scores['test_accuracy'].mean()
    print(f'Done. Mean accuracy: {mean_acc:.4f}')

print('\n✓ All models trained.')

In [ ]:
# Build a summary DataFrame with mean ± std for each metric
summary_rows = []

for model_name, scores in cv_results.items():
    row = {'Model': model_name}
    for metric in SCORING:
        key = f'test_{metric}'
        mean = scores[key].mean()
        std  = scores[key].std()
        # Nomes fixos para evitar inconsistência
        col_map = {
            'accuracy':          'Accuracy',
            'precision_macro':   'Precision Macro',
            'recall_macro':      'Recall Macro',
            'f1_macro':          'F1 Macro',
        }
        clean = col_map[metric]
        row[clean]            = mean
        row[f'{clean} Std']   = std
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')

display_cols = ['Accuracy', 'Precision Macro', 'Recall Macro', 'F1 Macro']
print('=== Cross-Validation Results (Mean over 5 folds) ===')
display(summary_df[display_cols].round(4).sort_values('Accuracy', ascending=False))

In [ ]:
# Identify best model by F1-Macro
best_model_name = summary_df['F1 Macro'].idxmax()
print(f'Best model by F1-Macro: {best_model_name}')
print(summary_df.loc[best_model_name, display_cols].round(4))

In [ ]:
# Plot: Accuracy comparison bar chart with error bars
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_sorted = summary_df.sort_values('Accuracy', ascending=False)

# Accuracy
axes[0].bar(
    models_sorted.index,
    models_sorted['Accuracy'],
    yerr=models_sorted['Accuracy Std'],
    capsize=5, color='steelblue', alpha=0.8
)
axes[0].set_title('Mean Accuracy (5-Fold GroupKFold)', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[0].patches, models_sorted['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# F1-Macro
models_f1 = summary_df.sort_values('F1 Macro', ascending=False)
axes[1].bar(
    models_f1.index,
    models_f1['F1 Macro'],
    yerr=models_f1['F1 Macro Std'],
    capsize=5, color='darkorange', alpha=0.8
)
axes[1].set_title('Mean F1-Score Macro (5-Fold GroupKFold)', fontsize=13)
axes[1].set_ylabel('F1 Macro')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[1].patches, models_f1['F1 Macro']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'model_comparison.png'), dpi=150)
plt.show()
print('Plot saved to processed directory.')

In [ ]:
# Retrain ALL models on the complete dataset and save them for Notebook 03
for model_name, pipeline in MODELS.items():
    pipeline.fit(X, y)
    safe_name = model_name.lower().replace(' ', '_')
    model_path = os.path.join(MODELS_DIR, f'{safe_name}.joblib')
    joblib.dump(pipeline, model_path)
    print(f'  Saved: {model_path}')

# Save the label encoder and feature columns for reproducibility
joblib.dump(le,           os.path.join(MODELS_DIR, 'label_encoder.joblib'))
joblib.dump(FEATURE_COLS, os.path.join(MODELS_DIR, 'feature_cols.joblib'))

# Save cv_results and summary for Notebook 03
summary_df.to_csv(os.path.join(PROCESSED_DIR, 'cv_summary.csv'))
joblib.dump(cv_results, os.path.join(PROCESSED_DIR, 'cv_results.joblib'))

print(f'\n✓ All models and metadata saved to {MODELS_DIR}')
print(f'✓ CV summary saved to {PROCESSED_DIR}cv_summary.csv')
print(f'\n→ Proceed to Notebook 03 for confusion matrices, ROC curves and full evaluation.')